# Lesson 12 – Self-Refinement

## Chapter 5 Connection

Chapter 5 of *Build a Reasoning Model from Scratch* introduces **self-refinement**.

The previous chapter showed an inference-time scaling method where we generate multiple responses and select the best one.

This chapter introduces a different strategy:

> Generate one response, evaluate it, revise it, and repeat.

For Weird AI, this means taking an initial parody and asking the model to improve it based on evaluation feedback.

This notebook is a concept lab. The applied programming assignment is completed separately in `src/weird_ai/refinement.py`.

## 1. From Best-of-N to Self-Refinement

Best-of-N generation is parallel:

```text
Generate Candidate A
Generate Candidate B
Generate Candidate C
        ↓
Evaluate all
        ↓
Choose best
```

Self-refinement is sequential:

```text
Generate Draft
    ↓
Evaluate Draft
    ↓
Revise Draft
    ↓
Evaluate Revision
    ↓
Revise Again
```

Both are inference-time scaling methods because they spend more compute during generation rather than changing the model's weights.

In [ ]:
initial_lyrics = '''
My database cried
Inside
The query died
Tonight
'''

print(initial_lyrics)


## 2. Critiquing Creative Output

Before we can refine a song, we need to identify what could be improved.

Look at the example above.

Possible issues:

- very short lines
- weak or inconsistent rhyme
- limited detail
- awkward rhythm
- unclear parody target

A human can critique these directly. An automated system needs scores or rules.

In [ ]:
def simple_line_report(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return {
        "line_count": len(lines),
        "average_line_length": sum(len(line) for line in lines) / max(1, len(lines)),
        "lines": lines,
    }

report = simple_line_report(initial_lyrics)
report


## 3. The Book's Scorer vs. Weird AI's Scorer

Chapter 5 begins with a rule-based scorer for math responses.

The book's scorer rewards things like:

- whether a final answer can be extracted
- whether the answer has the desired format
- whether the answer is concise

That makes sense for math reasoning.

Weird AI needs a different scorer. Our project cares about:

- rhyme quality
- syllable consistency
- lyric structure
- overall parody quality

This is an important AI engineering lesson:

> A scorer should match the task you actually care about.

In [ ]:
book_style_scores = {
    "has_final_answer": 2.0,
    "brevity_bonus": 0.7,
    "format_score": 1.0,
}

weird_ai_scores = {
    "rhyme_score": 0.62,
    "syllable_consistency_score": 0.55,
    "structure_score": 0.74,
    "overall_score": 0.64,
}

print("Book-style scorer:")
print(book_style_scores)

print("
Weird AI scorer:")
print(weird_ai_scores)


### Discussion

Why would a scorer that rewards brevity and final-answer format be a poor fit for parody lyrics?

## 4. Building a Refinement Prompt

A weak refinement prompt says:

```text
Make this better.
```

A stronger refinement prompt gives specific guidance:

```text
Revise these lyrics to improve rhyme and syllable consistency.
Preserve the topic and emotional tone.
Return only the revised lyrics.
```

The refinement prompt should include both the current text and the evaluation feedback.

In [ ]:
def build_demo_refinement_prompt(original_prompt, current_text, evaluation):
    return f'''
Original task:
{original_prompt}

Current lyrics:
{current_text}

Current evaluation:
- Overall score: {evaluation["overall_score"]}
- Rhyme score: {evaluation["rhyme_score"]}
- Syllable consistency: {evaluation["syllable_consistency_score"]}
- Structure score: {evaluation["structure_score"]}

Revise the lyrics to improve rhyme quality and syllable consistency.
Preserve the topic, tone, and approximate number of lines.
Return only the revised lyrics.
'''.strip()


original_prompt = "Write a short emo parody about database queries."

print(build_demo_refinement_prompt(original_prompt, initial_lyrics, weird_ai_scores))


## 5. Simulating One Refinement Step

In the real assignment, a model will generate the revised lyrics.

In this notebook, we will simulate the process with a manually written revision.

In [ ]:
revised_lyrics = '''
My database cried tonight
Every index lost the fight
Broken queries filled the air
Deadlocks whispered everywhere
'''

print(revised_lyrics)


## 6. Evaluating Before and After

A self-refinement loop should not automatically accept every revision.

Sometimes a revision gets worse.

A common strategy is:

```text
if revised_score > current_score:
    accept revision
else:
    keep previous version
```

In [ ]:
initial_eval = {
    "overall_score": 0.64,
    "rhyme_score": 0.62,
    "syllable_consistency_score": 0.55,
    "structure_score": 0.74,
}

revised_eval = {
    "overall_score": 0.81,
    "rhyme_score": 0.86,
    "syllable_consistency_score": 0.78,
    "structure_score": 0.80,
}

if revised_eval["overall_score"] > initial_eval["overall_score"]:
    selected = revised_lyrics
    print("Accepted revision.")
else:
    selected = initial_lyrics
    print("Rejected revision.")

print(selected)


## 7. Stopping Criteria

A refinement loop needs stopping rules.

Common stopping criteria:

- stop after a maximum number of iterations
- stop when a target score is reached
- stop when a revision fails to improve the score
- stop when the score improvement is very small

Without stopping rules, a refinement loop can waste compute or make the output worse.

In [ ]:
scores = [0.64, 0.81, 0.86, 0.86, 0.84]

target_score = 0.85
max_iterations = 4

for iteration, score in enumerate(scores):
    print(f"Iteration {iteration}: score = {score}")

    if score >= target_score:
        print("Stop: target score reached.")
        break

    if iteration >= max_iterations:
        print("Stop: maximum iterations reached.")
        break

    if iteration > 0 and score <= scores[iteration - 1]:
        print("Stop: no improvement.")
        break


## 8. When Refinement Makes Things Worse

Self-refinement is not guaranteed to improve output.

Possible problems:

- the model may drift away from the topic
- the model may over-optimize for the automated score
- the output may become less funny
- repeated revision may make the text generic
- the model may remove details that humans liked

This is why the project keeps the better-scoring version rather than blindly accepting each revision.

In [ ]:
version_a = '''
My database cried tonight
Every index lost the fight
Broken queries filled the air
Deadlocks whispered everywhere
'''

version_b = '''
Database night
Query fight
Data light
Code sight
'''

print("Version A:")
print(version_a)

print("
Version B:")
print(version_b)

print("
Question: Which version might score better automatically? Which version do you prefer as a human?")


## 9. Designing the Weird AI Refinement Loop

The assignment will ask you to implement this pipeline:

```text
Initial Lyrics
      ↓
Evaluate
      ↓
Build Refinement Prompt
      ↓
Generate Revision
      ↓
Evaluate Revision
      ↓
Keep Better Version
      ↓
Repeat Until Stop
```

The result should keep track of every iteration so we can inspect what happened.

In [ ]:
# Design notes for your implementation

preserve = ["topic", "tone", "approximate line count"]
focus = ["stronger rhymes", "more consistent syllables", "clearer parody target"]
stopping_rules = ["maximum iterations", "target score", "no improvement"]

print("Preserve:", preserve)
print("Focus:", focus)
print("Stopping rules:", stopping_rules)


## 10. Reflection

1. How is self-refinement different from best-of-N generation?
2. Why does self-refinement increase inference-time compute?
3. How is the book's heuristic scorer different from Weird AI's project scorer?
4. Why should a task-specific scorer be preferred over a generic scorer?
5. Why might a higher automated score not always mean a better parody?
6. When should a refinement loop stop?
7. How could best-of-N and self-refinement be combined?